In [0]:
from pyspark.sql import functions as F

# Run 1
run1_patients = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("dbfs:/FileStore/healthcare/run_1_dev/patients.csv")
    .withColumn("run_id", F.lit("run_1_dev"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

# Run 2
run2_patients = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("dbfs:/FileStore/healthcare/run_2_prod/patients.csv")
    .withColumn("run_id", F.lit("run_2_prod"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

In [0]:
print("Run 1 count:", run1_patients.count())
print("Run 2 count:", run2_patients.count())

Run 1 count: 1159
Run 2 count: 11577


In [0]:
run1_patients.select("Id").show(5, False)
run2_patients.select("Id").show(5, False)

+------------------------------------+
|Id                                  |
+------------------------------------+
|9cd86fc2-45a2-293d-a78d-4bf3c38b25e9|
|c2d3d937-3310-1250-2faa-d5adf7db6317|
|4bec861c-d389-4dd7-ebff-ee35d5a3f68f|
|5865e962-cd23-4dbb-a8b7-639ca10f7710|
|c60092f4-facc-bd60-c5a9-c6739e3c80e6|
+------------------------------------+
only showing top 5 rows

+------------------------------------+
|Id                                  |
+------------------------------------+
|fbfc3ffd-0b7f-1606-2e5a-44eb4a209df2|
|b4ce54cc-edcf-051c-ce0f-c1fe3455f545|
|ae9e879f-ec52-a728-f873-c1314d7f346d|
|6463b5d8-9cf1-e471-119b-c0553dbf5946|
|6836e7b2-8647-dacf-45e8-3921c663575d|
+------------------------------------+
only showing top 5 rows



In [0]:
overlap_patients = (
    run1_patients.select("Id").distinct()
    .join(run2_patients.select("Id").distinct(), "Id", "inner")
)

print("Overlapping patient IDs:", overlap_patients.count())

Overlapping patient IDs: 0


In [0]:
print("Current bronze patients:", spark.table("healthcare_project.bronze_patients").count())
print("Current bronze conditions:", spark.table("healthcare_project.bronze_conditions").count())
print("Current bronze encounters:", spark.table("healthcare_project.bronze_encounters").count())
print("Current bronze medications:", spark.table("healthcare_project.bronze_medications").count())
print("Current bronze procedures:", spark.table("healthcare_project.bronze_procedures").count())
print("Current bronze providers:", spark.table("healthcare_project.bronze_providers").count())

Current bronze patients: 1159
Current bronze conditions: 43447
Current bronze encounters: 70028
Current bronze medications: 60636
Current bronze procedures: 192060
Current bronze providers: 810


In [0]:
spark.table("healthcare_project.bronze_patients").printSchema()

root
 |-- Id: string (nullable = true)
 |-- BIRTHDATE: date (nullable = true)
 |-- DEATHDATE: date (nullable = true)
 |-- SSN: string (nullable = true)
 |-- DRIVERS: string (nullable = true)
 |-- PASSPORT: string (nullable = true)
 |-- PREFIX: string (nullable = true)
 |-- FIRST: string (nullable = true)
 |-- MIDDLE: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- SUFFIX: string (nullable = true)
 |-- MAIDEN: string (nullable = true)
 |-- MARITAL: string (nullable = true)
 |-- RACE: string (nullable = true)
 |-- ETHNICITY: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- BIRTHPLACE: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- FIPS: long (nullable = true)
 |-- ZIP: long (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- HEALTHCARE_EXPENSES: double (nullable = true)
 |-- HEALT

In [0]:
run2_patients_to_bronze.printSchema()

root
 |-- Id: string (nullable = true)
 |-- BIRTHDATE: date (nullable = true)
 |-- DEATHDATE: date (nullable = true)
 |-- SSN: string (nullable = true)
 |-- DRIVERS: string (nullable = true)
 |-- PASSPORT: string (nullable = true)
 |-- PREFIX: string (nullable = true)
 |-- FIRST: string (nullable = true)
 |-- MIDDLE: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- SUFFIX: string (nullable = true)
 |-- MAIDEN: string (nullable = true)
 |-- MARITAL: string (nullable = true)
 |-- RACE: string (nullable = true)
 |-- ETHNICITY: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- BIRTHPLACE: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- FIPS: integer (nullable = true)
 |-- ZIP: integer (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- HEALTHCARE_EXPENSES: double (nullable = true)
 |--

In [0]:
from pyspark.sql import functions as F

schema_name = "healthcare_project"
run_id = "run_2_prod"
base_path = f"dbfs:/FileStore/healthcare/{run_id}/"

file_names = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "conditions": "conditions.csv",
    "procedures": "procedures.csv",
    "medications": "medications.csv",
    "providers": "providers.csv"
}

def append_incremental_run(table_name, file_name, run_id, schema_name="healthcare_project"):
    target_table = f"{schema_name}.bronze_{table_name}"
    file_path = base_path + file_name

    print(f"\nProcessing {table_name} from {file_path}")

    # Read new run
    new_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    # Add standard metadata columns for incremental ingestion
    if "run_id" not in new_df.columns:
        new_df = new_df.withColumn("run_id", F.lit(run_id))

    if "ingestion_timestamp" not in new_df.columns:
        new_df = new_df.withColumn("ingestion_timestamp", F.current_timestamp())

    if "source_file" not in new_df.columns:
        new_df = new_df.withColumn("source_file", F.input_file_name())

    if "data_source" not in new_df.columns:
        new_df = new_df.withColumn("data_source", F.lit("synthea"))

    # Get existing bronze schema
    existing_schema = spark.table(target_table).schema
    existing_cols = [field.name for field in existing_schema]

    # Add any missing columns from the existing table schema
    for field in existing_schema:
        if field.name not in new_df.columns:
            new_df = new_df.withColumn(field.name, F.lit(None).cast(field.dataType))

    # Cast matching columns to target schema types
    for field in existing_schema:
        if field.name in new_df.columns:
            new_df = new_df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    # Keep only columns that exist in target table, in exact order
    new_df = new_df.select(existing_cols)

    before_count = spark.table(target_table).count()

    # Append safely
    new_df.write.mode("append").saveAsTable(target_table)

    after_count = spark.table(target_table).count()
    appended_rows = after_count - before_count

    print(f"Appended {appended_rows} rows into {target_table}")
    print(f"Before count: {before_count}")
    print(f"After count : {after_count}")


# Run for all six bronze tables
for table_name, file_name in file_names.items():
    append_incremental_run(table_name, file_name, run_id, schema_name)


Processing patients from dbfs:/FileStore/healthcare/run_2_prod/patients.csv
Appended 11577 rows into healthcare_project.bronze_patients
Before count: 1159
After count : 12736

Processing encounters from dbfs:/FileStore/healthcare/run_2_prod/encounters.csv
Appended 672336 rows into healthcare_project.bronze_encounters
Before count: 70028
After count : 742364

Processing conditions from dbfs:/FileStore/healthcare/run_2_prod/conditions.csv
Appended 417971 rows into healthcare_project.bronze_conditions
Before count: 43447
After count : 461418

Processing procedures from dbfs:/FileStore/healthcare/run_2_prod/procedures.csv
Appended 1873828 rows into healthcare_project.bronze_procedures
Before count: 192060
After count : 2065888

Processing medications from dbfs:/FileStore/healthcare/run_2_prod/medications.csv
Appended 562536 rows into healthcare_project.bronze_medications
Before count: 60636
After count : 623172

Processing providers from dbfs:/FileStore/healthcare/run_2_prod/providers.csv